# 5. Descriptive statistics and topic breakdown

Build the Paper 1 sample flow, discussion/month/topic tables, feature summaries, and figures from the shared 2025 model contract. The primary topic is the uncollapsed `section_1`; `section_2` and `section_3` are retained in the hierarchy audit. Missing `section_1` values are reported as `Unknown/other`. Audience-selected summaries average all ten deterministic tie draws.

The default scope is `all`. Set `COMMENTGAP_MODEL_SCOPES=all,root` to add the root-comment appendix tables. This stage reads the frozen model data but writes only below `model_output/selection_2025/paper1/descriptives/`.

In [ ]:
from pathlib import Path
import os

from commentgap_analysis.paper1_descriptives import run_descriptive_analysis

scope_text = os.getenv("COMMENTGAP_MODEL_SCOPES", "all")
SCOPES = tuple(dict.fromkeys(part.strip() for part in scope_text.split(",") if part.strip()))
if not SCOPES or not set(SCOPES).issubset({"all", "root"}):
    raise ValueError(f"Invalid COMMENTGAP_MODEL_SCOPES={scope_text!r}")

MODEL_DATA_ROOT = Path(os.getenv("COMMENTGAP_MODEL_DATA_ROOT", "model_output/selection_2025/model_data"))
DATA_ROOT = Path(os.getenv("COMMENTGAP_DATA_ROOT", "data/scrape_2025"))
OUTPUT_ROOT = Path(os.getenv("COMMENTGAP_DESCRIPTIVES_ROOT", "model_output/selection_2025/paper1/descriptives"))
THREADS = int(os.getenv("COMMENTGAP_DESCRIPTIVE_THREADS", "4"))
{"scopes": SCOPES, "model_data": str(MODEL_DATA_ROOT), "output": str(OUTPUT_ROOT)}

In [ ]:
manifest = run_descriptive_analysis(
    model_data_root=MODEL_DATA_ROOT,
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    scopes=SCOPES,
    threads=THREADS,
    make_figures=True,
)
manifest

## Key results

The cells below surface the main sample counts, topic composition, and largest curator–audience feature contrasts. The tables are read from the canonical stage outputs so that the displayed results match the manifest and saved figures.

In [ ]:
import pandas as pd
from IPython.display import Image, display

sample_flow = pd.read_csv(OUTPUT_ROOT / "sample_flow.csv")
topic_summary = pd.read_csv(OUTPUT_ROOT / "topic_summary.csv")
feature_summary = pd.read_csv(OUTPUT_ROOT / "feature_summary.csv")

display(
    sample_flow.style
    .format({"n_articles": "{:,.0f}", "n_comment_rows": "{:,.0f}", "n_candidates": "{:,.0f}", "n_picks": "{:,.0f}"}, na_rep="—")
    .hide(axis="index")
    .set_caption("Sample flow")
)

In [ ]:
primary_scope = "all" if "all" in SCOPES else SCOPES[0]
primary_topics = topic_summary.query(
    "scope == @primary_scope and analysis_partition == 'all_partitions'"
).copy()
headline = pd.DataFrame({
    "eligible discussions": [primary_topics["n_articles"].sum()],
    "candidate comments": [primary_topics["n_candidates"].sum()],
    "curator picks": [primary_topics["n_curator_picks"].sum()],
    "primary topics": [primary_topics["primary_topic"].nunique()],
    "mean candidates/discussion": [primary_topics["n_candidates"].sum() / primary_topics["n_articles"].sum()],
})
top_topics = primary_topics.nlargest(15, "n_articles")[[
    "primary_topic", "n_articles", "n_candidates", "picks_per_article_mean", "reply_share_mean"
]]
display(
    headline.style.format({
        "eligible discussions": "{:,.0f}", "candidate comments": "{:,.0f}",
        "curator picks": "{:,.0f}", "primary topics": "{:,.0f}",
        "mean candidates/discussion": "{:,.1f}",
    }).hide(axis="index").set_caption(f"Headline results ({primary_scope})"),
    top_topics.style.format({
        "n_articles": "{:,.0f}", "n_candidates": "{:,.0f}",
        "picks_per_article_mean": "{:.2f}", "reply_share_mean": "{:.1%}",
    }).hide(axis="index").set_caption("15 largest topics"),
)
display(Image(filename=str(OUTPUT_ROOT / "figures" / f"{primary_scope}_topic_composition.png"), width=850))

In [ ]:
feature_contrasts = (
    feature_summary.query("scope == @primary_scope and feature_source == 'transformed_model_predictor'")
    .dropna(subset=["curator_minus_audience_sd"])
    .assign(abs_contrast=lambda frame: frame["curator_minus_audience_sd"].abs())
    .nlargest(10, "abs_contrast")
    [["label", "feature_group", "curator_mean", "audience_mean", "curator_minus_audience_sd"]]
)
display(
    feature_contrasts.style.format({
        "curator_mean": "{:.3f}", "audience_mean": "{:.3f}",
        "curator_minus_audience_sd": "{:+.3f}",
    }).hide(axis="index").set_caption("Largest curator–audience feature contrasts (candidate SDs)")
)
display(Image(filename=str(OUTPUT_ROOT / "figures" / f"{primary_scope}_selected_feature_contrasts.png"), width=850))

## Output contract

The manifest is the completion watermark. Core outputs are `article_topics.parquet`, `discussion_descriptives.parquet`, `sample_flow.csv`, `collection_qa_status.csv`, `monthly_summary.csv`, `topic_summary.csv`, `topic_hierarchy_audit.csv`, and `feature_summary.csv`, plus PNG/PDF figures. Vote fields are collection-time snapshots; the shared model contract retains net `relative_votes`, not separate positive and negative totals.